# Python File Handling: CSV Files

A practical guide to reading and writing comma-separated tabular data with Python's built-in `csv` module.

**Learning goals**
- Understand CSV rows, headers, quoting, and why a CSV remains plain text.
- Use `csv.reader` / `csv.writer` for list-based rows.
- Use `csv.DictReader` / `csv.DictWriter` for named columns.
- Convert strings to numbers, filter data, append records, and handle malformed input.

The examples create files under `csv_data/` in the notebook's current working directory. They follow the reference PDF's guidance to use `newline=""` and avoid splitting rows manually on commas.

## 1. What is CSV?

CSV means comma-separated values. Each row commonly represents one record and the first row commonly names the columns. A CSV file is text with conventions for representing table cells. Fields containing commas, quotes, or line breaks need quoting, which the `csv` module handles correctly.

Do not parse general CSV by calling `line.split(",")`: a valid quoted field can itself contain a comma.

In [1]:
from pathlib import Path
import csv

DATA_DIR = Path.cwd() / "csv_data"
DATA_DIR.mkdir(exist_ok=True)
STUDENTS_PATH = DATA_DIR / "students.csv"

rows = [
    ["Name", "Age", "Marks", "Comment"],
    ["Rahul", 18, 85, "Good progress"],
    ["Priya", 19, 92, "Excellent, keep it up"],
    ["Aman", 18, 78, 'Needs "extra" practice'],
]

print("Working folder:", Path.cwd())

Working folder: c:\Users\ashir\OneDrive\Desktop\Mirai\AKTU\Python\File_handling


## 2. Write rows with `csv.writer`

Open CSV files with `newline=""`; this lets the CSV module manage line endings and avoids extra blank rows on some platforms. `writerow()` writes one row and `writerows()` writes an iterable of rows.

In [2]:
with open(STUDENTS_PATH, "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerows(rows)

print(STUDENTS_PATH.read_text(encoding="utf-8"))

Name,Age,Marks,Comment
Rahul,18,85,Good progress
Priya,19,92,"Excellent, keep it up"
Aman,18,78,"Needs ""extra"" practice"



## 3. Read rows with `csv.reader`

Each row returned by `csv.reader` is a list of strings. Even values written as numbers are read as strings, so convert them before arithmetic.

In [3]:
with open(STUDENTS_PATH, "r", newline="", encoding="utf-8") as file:
    reader = csv.reader(file)
    header = next(reader)
    print("Columns:", header)
    for row in reader:
        print(row, "->", type(row[1]).__name__)

with open(STUDENTS_PATH, "r", newline="", encoding="utf-8") as file:
    reader = csv.reader(file)
    header = next(reader)
    marks_index = header.index("Marks")
    marks = [int(row[marks_index]) for row in reader]

print("Marks:", marks)
print("Average:", sum(marks) / len(marks))

Columns: ['Name', 'Age', 'Marks', 'Comment']
['Rahul', '18', '85', 'Good progress'] -> str
['Priya', '19', '92', 'Excellent, keep it up'] -> str
['Aman', '18', '78', 'Needs "extra" practice'] -> str
Marks: [85, 92, 78]
Average: 85.0


## 4. Dictionary-based CSV: `DictWriter`

`DictWriter` maps dictionary keys to columns. Passing `fieldnames` fixes the output column order. `writeheader()` writes the header row. Choose `extrasaction="raise"` (the default) to catch unexpected keys.

In [4]:
FIELDNAMES = ["Name", "Age", "Marks", "Comment"]
student_records = [
    {"Name": "Meera", "Age": 20, "Marks": 88, "Comment": "On track"},
    {"Name": "Dev", "Age": 19, "Marks": 95, "Comment": "Great work"},
]

with open(STUDENTS_PATH, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=FIELDNAMES)
    writer.writeheader()
    writer.writerows(student_records)

print(STUDENTS_PATH.read_text(encoding="utf-8"))

Name,Age,Marks,Comment
Meera,20,88,On track
Dev,19,95,Great work



## 5. Read named columns with `DictReader`

A `DictReader` uses the first row as field names by default. Each subsequent record behaves like a dictionary. Values still arrive as strings.

In [5]:
with open(STUDENTS_PATH, "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for row in reader:
        print(row["Name"], row["Marks"], type(row["Marks"]).__name__)

with open(STUDENTS_PATH, "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    high_scorers = [row["Name"] for row in reader if int(row["Marks"]) >= 90]

print("Marks >= 90:", high_scorers)

Meera 88 str
Dev 95 str
Marks >= 90: ['Dev']


## 6. Appending a CSV record

Appending a record is straightforward if the file already has its header. Avoid writing a second header. For a reusable program, check whether the file exists or is empty before deciding to write the header.

In [6]:
import os

append_path = DATA_DIR / "attendance.csv"
new_record = {"Name": "Kiran", "Present": "yes"}
fields = ["Name", "Present"]
needs_header = not append_path.exists() or append_path.stat().st_size == 0

with open(append_path, "a", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=fields)
    if needs_header:
        writer.writeheader()
    writer.writerow(new_record)

print(append_path.read_text(encoding="utf-8"))

Name,Present
Kiran,yes



## 7. CSV dialects and delimiter options

CSV conventions vary. If a file uses semicolons, set `delimiter=";"`. The default `excel` dialect is suitable for common spreadsheet CSV files. The CSV module also handles quoted delimiters and escaped quotes.

In [7]:
semicolon_path = DATA_DIR / "products_semicolon.csv"
with open(semicolon_path, "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file, delimiter=";")
    writer.writerow(["Product", "Price"])
    writer.writerow(["Tea; premium blend", "120.50"])

with open(semicolon_path, "r", newline="", encoding="utf-8") as file:
    for row in csv.reader(file, delimiter=";"):
        print(row)

['Product', 'Price']
['Tea; premium blend', '120.50']


## 8. Validate and convert input data

CSV does not enforce types or required columns. Validate them in your program and report the row number when data is not usable. This example rejects blank names and non-integer marks.

In [8]:
def load_valid_marks(path):
    valid = []
    with open(path, "r", newline="", encoding="utf-8") as file:
        reader = csv.DictReader(file)
        required = {"Name", "Marks"}
        if not reader.fieldnames or not required.issubset(reader.fieldnames):
            raise ValueError(f"CSV must include columns: {sorted(required)}")
        for line_number, row in enumerate(reader, start=2):
            name = (row.get("Name") or "").strip()
            try:
                score = int(row["Marks"])
            except (TypeError, ValueError) as error:
                raise ValueError(f"Invalid Marks at CSV line {line_number}") from error
            if not name:
                raise ValueError(f"Name is blank at CSV line {line_number}")
            valid.append((name, score))
    return valid

print(load_valid_marks(STUDENTS_PATH))

[('Meera', 88), ('Dev', 95)]


## 9. Common CSV mistakes

- Forgetting `import csv`.
- Omitting `newline=""` when opening a CSV.
- Assuming numbers automatically become `int` or `float`.
- Using the wrong working directory or filename.
- Splitting a line on commas by hand.
- Appending a duplicate header.
- Assuming every CSV uses a comma delimiter or the same text encoding.

**Choosing an API:** use `reader` / `writer` when lists are natural; use `DictReader` / `DictWriter` when column names make code clearer.

## 10. Summary and practice

1. Why does `csv.reader` return strings for numeric-looking cells?
2. What does `writeheader()` do?
3. Why can `split(",")` misread a CSV row?
4. Modify the analysis to find the highest mark and its student's name.
5. Write a new CSV containing only students scoring at least 85, with a header row.